# ДЗ 6. Очистка orders.csv в Pandas
Срок: **до воскресенья 22.11, 24:00**.
Сдача: папка `hw06` репозитория `dsa-homework` (`hw06.ipynb`): `git add hw06` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать.

Инструменты: всё с уроков 1–6. `groupby`, `pivot_table`, `merge` пока не используем (урок 7). Небольшой цикл по городам или месяцам допустим. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
s = pd.Series([1, None, 3])
print(s.isna().sum(), s.fillna(0).mean())
# 2
print(pd.Series([" a", "A ", "b"]).str.strip().str.lower().nunique())
# 3
df = pd.DataFrame({"k": [1, 1, 2]})
df.drop_duplicates()
print(len(df))
# 4
print(pd.to_numeric(pd.Series(["1 000", "5"]).str.replace(" ", "")).sum())
# 5
d = pd.to_datetime(pd.Series(["2026-09-30 21:00", "2026-10-01 08:00"]))
print(d.dt.month.tolist())
```

1. `1 1.3333333333333333`: один пропуск, а после заполнения нулем среднее (1+0+3)/3
2. `2`: пробелы убираются, регистр выравнивается и остаются значения `a` и `b`
3. `3`: `drop_duplicates()` возвращает новую таблицу и не изменяет `df` без присваивания
4. `1005`: пробел убрали, строки преобразовали в числа, а потом сложили
5. `[9, 10]`: даты распознаны как datetime, а `.dt.month` получает месяцы

In [5]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

Saving orders.csv to orders.csv
orders.csv на месте: True


## [База] 1. Функция очистки
Напишите функцию `clean(path)`, которая читает orders.csv и делает все шаги урока: суммы в числа, города через словарь, дубли, даты, без TEST. Отменённые заказы **не** удаляйте: они нужны для доли отмен.
После каждого шага функция печатает, сколько строк осталось. Проверки внутри функции: у `amount` нет NaN, у `city` нет NaN, `order_id` уникален.
Сколько строк в результате?

In [6]:
import pandas as pd

CITY = {"алматы": "Алматы", "almaty": "Алматы",
        "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент",
        "караганда": "Караганда", "актобе": "Актобе", "атырау": "Атырау"}

def clean(path):
    data = pd.read_csv(path)
    print("Изначально строк: ", len(data))
    data["amount"] = pd.to_numeric(data["amount"].str.replace(" ", ""))
    print("Строк после преобразования сумм: ", len(data), "пропусков: ", data["amount"].isna().sum())
    data["city"] = data["city"].str.strip().str.lower().map(CITY)
    print("Строк после приведения городов: ", len(data), "пропусков: ", data["city"].isna().sum())
    data = data.drop_duplicates().copy()
    print("Строк после удаления дублей: ", len(data))
    data["order_date"] = pd.to_datetime(data["order_date"])
    print("Строк после преобразования дат: ", len(data))
    data = data.loc[data["customer_id"] != "TEST"].copy()
    print("Строк после удаления тестовых данных: ", len(data))
    print("Проверка amount на NaN: ", data["amount"].isna().sum())
    print("Проверка city на NaN: ", data["city"].isna().sum())
    print("Проверка order_id на уникальность: ", data["order_id"].is_unique)
    return data

clean_df = clean("orders.csv")

Изначально строк:  5463
Строк после преобразования сумм:  5463 пропусков:  0
Строк после приведения городов:  5463 пропусков:  0
Строк после удаления дублей:  5283
Строк после преобразования дат:  5283
Строк после удаления тестовых данных:  5280
Проверка amount на NaN:  0
Проверка city на NaN:  0
Проверка order_id на уникальность:  True


## [База] 2. Выручка по чистым данным
Выручка оплаченных заказов за сентябрь и октябрь и изменение в %. Затем то же для каждого из 6 городов (цикл по городам).
Совпадает ли с таблицей «город × месяц» урока 4? Какой город падает сильнее всех?

In [13]:
paid = clean_df["status"] == "оплачен"
sep = clean_df["order_date"].dt.month == 9
october = clean_df["order_date"].dt.month == 10
sep_amount = clean_df.loc[paid & sep, "amount"].sum()
oct_amount = clean_df.loc[paid & october, "amount"].sum()
print(f"Сентябрь: {sep_amount:,.0f}, Октябрь: {oct_amount:,.0f}")
print(f"Изменение: {(oct_amount / sep_amount - 1):.1%}")
print("Изменение по городам (сентябрь -> октябрь):")
worst_city = ""
worst_change = 0
for city in ["Алматы", "Астана", "Шымкент", "Караганда", "Актобе", "Атырау"]:
    city_sep = clean_df.loc[paid & sep & (clean_df["city"] == city), "amount"].sum()
    city_oct = clean_df.loc[paid & october & (clean_df["city"] == city), "amount"].sum()
    change = city_oct / city_sep - 1
    print(f"{city}: {city_sep:,.0f} -> {city_oct:,.0f} ({change:.1%})")
    if change < worst_change:
        worst_change, worst_city = change, city
print(worst_city, f"({worst_change:.1%})")

Сентябрь: 35,243,410, Октябрь: 34,419,520
Изменение: -2.3%
Изменение по городам (сентябрь -> октябрь):
Алматы: 14,501,070 -> 14,643,020 (1.0%)
Астана: 9,034,220 -> 8,850,530 (-2.0%)
Шымкент: 4,558,650 -> 4,044,010 (-11.3%)
Караганда: 3,648,520 -> 3,371,680 (-7.6%)
Актобе: 2,576,960 -> 2,540,420 (-1.4%)
Атырау: 923,990 -> 969,860 (5.0%)
Шымкент (-11.3%)


## [База] 3. Журнал очистки для менеджера
Markdown-таблица: шаг очистки → сколько строк убрал → как изменилось «падение выручки» (от −22,7% до −2,3%).
Одной фразой для менеджера: какой шаг сильнее всего исказил его цифру?

In [14]:
raw = pd.read_csv("orders.csv")
raw["amount"] = pd.to_numeric(raw["amount"].str.replace(" ", ""))

def month_change(data):
    paid_rows = data["status"] == "оплачен"
    s = data.loc[paid_rows & (data["order_date"].str[:7] == "2026-09"), "amount"].sum()
    o = data.loc[paid_rows & (data["order_date"].str[:7] == "2026-10"), "amount"].sum()
    return 100 * (o / s - 1)

without_duplicates = raw.drop_duplicates().copy()
without_test = without_duplicates.loc[without_duplicates["customer_id"] != "TEST"].copy()
steps = [("Исходный файл", raw), ("Без полных дублей", without_duplicates),
         ("Без TEST", without_test)]

print("Шаг | Строк осталось | Строк удалено | Изменение октябрь/сентябрь")

previous = len(raw)
for name, data in steps:
    removed = previous - len(data)
    print(f"{name} | {len(data)} | {removed} | {month_change(data):.1f}%")
    previous = len(data)

Шаг | Строк осталось | Строк удалено | Изменение октябрь/сентябрь
Исходный файл | 5463 | 0 | -22.7%
Без полных дублей | 5283 | 180 | -19.6%
Без TEST | 5280 | 3 | -2.3%


## [Стандарт] 4. Пропуски в сроках доставки
Сколько пропусков в `delivery_days` по чистым данным? В каких городах и статусах их больше?
Доля доставки за 2 дня в Шымкенте: (а) `dropna`, (б) `fillna(медиана)`, (в) `fillna(0)`. Какой вариант честный и почему другие вводят в заблуждение?

In [15]:
missing = clean_df["delivery_days"].isna()
print("Пропусков в delivery_days: ", missing.sum())
print("Пропуски по городам:")
print(clean_df.loc[missing, "city"].value_counts())
print("Пропуски по статусам:")
print(clean_df.loc[missing, "status"].value_counts())
d = clean_df.loc[clean_df["city"] == "Шымкент", "delivery_days"]
print(f"Шымкент, без пустых значений: {(d.dropna() <= 2).mean():.1%}")
print(f"Шымкент, медиана: {(d.fillna(d.median()) <= 2).mean():.1%}")
print(f"Шымкент, с заменой на нулевые значения: {(d.fillna(0) <= 2).mean():.1%}")

Пропусков в delivery_days:  158
Пропуски по городам:
city
Алматы       63
Астана       40
Караганда    24
Шымкент      22
Атырау        5
Актобе        4
Name: count, dtype: int64
Пропуски по статусам:
status
оплачен    144
отменён     14
Name: count, dtype: int64
Шымкент, без пустых значений: 29.8%
Шымкент, медиана: 29.0%
Шымкент, с заменой на нулевые значения: 31.8%


## [Стандарт] 5. Исправьте код от AI
AI написал «очистку» и получил 10 городов вместо 6, а строк осталось 5 463. В коде **две** ошибки. Найдите их и исправьте.
```python
df = pd.read_csv("orders.csv")
df.drop_duplicates()
df["city"] = df["city"].str.lower()
print(df["city"].nunique(), len(df))
```

In [16]:
df_ai = pd.read_csv("orders.csv")
print(df_ai["city"].str.lower().nunique(), "городов;", len(df_ai), "строк")
df_ai = df_ai.drop_duplicates().copy()
df_ai["city"] = df_ai["city"].str.strip().str.lower().map(CITY)
print(df_ai["city"].nunique(), "городов;", len(df_ai), "строк")

10 городов; 5463 строк
6 городов; 5283 строк


Полные дубли исключены и уникализированы все написания городов

## [Стандарт] 6. Дни недели
По чистым данным: сколько заказов в каждый день недели (`.dt.day_name()` и `value_counts`)? В какой день заказов меньше всего и на сколько процентов меньше, чем в самый загруженный?
Что это значит для графика работы курьеров?

In [19]:
day_counts = clean_df["order_date"].dt.day_name().value_counts()
print("Заказов в каждый день недели:")
print(day_counts)
day_sorted = day_counts.sort_values()
least_day = day_sorted.index[0]
most_day = day_sorted.index[-1]
percent_less = (1 - day_counts.min() / day_counts.max()) * 100
print(f"Меньше всего заказов в: {least_day} ({day_counts.min()} заказов)")
print(f"Разница в % между самым самым загруженным днем и самым менее загруженным: {percent_less:.1f}")

Заказов в каждый день недели:
order_date
Thursday     786
Tuesday      785
Friday       781
Wednesday    774
Saturday     773
Sunday       697
Monday       684
Name: count, dtype: int64
Меньше всего заказов в: Monday (684 заказов)
Разница в % между самым самым загруженным днем и самым менее загруженным: 13.0


Это значит необходимость равномерного распределения нагрузки курьеров по дням

## [Челлендж] 7. Когда сломалась доставка в Шымкенте
Средний срок доставки в Шымкенте в сентябре и в октябре. Найдите первую дату, с которой срок доставки в Шымкенте не бывает меньше 5 дней (фильтр + `sort_values` по дате).
Проверьте, что в других городах в эту дату ничего не изменилось. Напишите менеджеру: что случилось, когда и что проверить в первую очередь.

In [1]:
# 7

## [Челлендж] 8. Надёжный перевод сумм
Напишите функцию `to_amount(s)` для Series строк, которая понимает `"15 990"`, `"15990.0"`, `"1 200,50"` (запятая как разделитель дробной части) и превращает всё остальное (`"нет данных"`) в NaN.
Проверьте на `pd.Series(["15 990", "15990.0", "1 200,50", "нет данных"])` и на столбце `amount` из orders.csv: сумма должна совпасть с уроком.

In [2]:
# 8